In [1]:
!pip install -U -q transformers accelerate pillow tqdm sentencepiece protobuf
!pip uninstall -y pillow PIL
!pip install -q --no-cache-dir --force-reinstall pillow==10.4.0
!pip install -U -q torchao peft accelerate transformers bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 31.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.34.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, bu

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import zipfile
import os

DRIVE_ROOT = Path("/content/drive/MyDrive")
ZIP_PATH = DRIVE_ROOT / "kagglefinal" / "pixels-to-predictions.zip"

DATA_DIR = Path("/content/pixels-to-predictions")

print("Zip exists:", ZIP_PATH.exists())
print("Zip path:", ZIP_PATH)

Zip exists: True
Zip path: /content/drive/MyDrive/kagglefinal/pixels-to-predictions.zip


In [4]:
if not DATA_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("Unzipped to:", DATA_DIR)
else:
    print("Already unzipped:", DATA_DIR)

print("Files:")
for p in DATA_DIR.iterdir():
    print(" -", p)

import json
import pandas as pd
import numpy as np
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)
print("Sample submission:", sample_submission.shape)


from pathlib import Path

def resolve_image_path(image_path):
    image_path = Path(image_path)

    possible_paths = [
        DATA_DIR / image_path,
        DATA_DIR / "images" / image_path,
        DATA_DIR / "images" / image_path.name,
        DATA_DIR / "images" / image_path.parent / image_path.name,
    ]

    for p in possible_paths:
        if p.exists():
            return p

    raise FileNotFoundError(f"Could not find image: {image_path}")

print(resolve_image_path(train_df.iloc[0]["image_path"]))

Unzipped to: /content/pixels-to-predictions
Files:
 - /content/pixels-to-predictions/sample_submission.csv
 - /content/pixels-to-predictions/val.csv
 - /content/pixels-to-predictions/test.csv
 - /content/pixels-to-predictions/images
 - /content/pixels-to-predictions/train.csv
Train: (3109, 15)
Val: (1048, 15)
Test: (1008, 13)
Sample submission: (1008, 2)
/content/pixels-to-predictions/images/images/train/train_07667.png


In [5]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_NAME = "HuggingFaceTB/SmolVLM-500M-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

model.eval()
print("Model loaded!")

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Model loaded!


In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,568,256 || all params: 517,050,560 || trainable%: 1.8505


In [7]:
from torch.utils.data import Dataset
from PIL import Image
import torch
import pandas as pd

def short_text(x, max_chars=500):
    if pd.isna(x):
        return ""
    return str(x)[:max_chars]


def build_user_prompt(row):
    choices_text = ""
    for i, choice in enumerate(row["choices"]):
        choices_text += f"{i}. {choice}\n"

    hint = short_text(row["hint"], 300) if "hint" in row.index else ""
    lecture = short_text(row["lecture"], 400) if "lecture" in row.index else ""

    subject = row["subject"] if "subject" in row.index and pd.notna(row["subject"]) else ""
    grade = row["grade"] if "grade" in row.index and pd.notna(row["grade"]) else ""
    topic = row["topic"] if "topic" in row.index and pd.notna(row["topic"]) else ""
    category = row["category"] if "category" in row.index and pd.notna(row["category"]) else ""
    skill = row["skill"] if "skill" in row.index and pd.notna(row["skill"]) else ""

    prompt = f"""Look at the image and answer the science multiple-choice question.

Metadata:
Subject: {subject}
Grade: {grade}
Topic: {topic}
Category: {category}
Skill: {skill}

Question:
{row['question']}

Hint:
{hint}

Context:
{lecture}

Choices:
{choices_text}

Return only the correct answer index from 0 to {row['num_choices'] - 1}.
"""
    return prompt


def build_prompt_text(row):
    prompt = build_user_prompt(row)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    return processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )


def build_full_train_text(row):
    prompt_text = build_prompt_text(row)
    answer = str(int(row["answer"]))
    return prompt_text + answer


class ScienceVLMDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = resolve_image_path(row["image_path"])
        image = Image.open(img_path).convert("RGB")

        prompt_text = build_prompt_text(row)
        full_text = build_full_train_text(row)

        prompt_inputs = processor(
            text=prompt_text,
            images=[image],
            return_tensors="pt",
            padding=False,
        )

        full_inputs = processor(
            text=full_text,
            images=[image],
            return_tensors="pt",
            padding=False,
        )

        item = {}
        for k, v in full_inputs.items():
            item[k] = v.squeeze(0)

        labels = item["input_ids"].clone()

        # Mask prompt tokens so loss is only on answer token
        prompt_len = prompt_inputs["input_ids"].shape[1]
        labels[:prompt_len] = -100

        if processor.tokenizer.pad_token_id is not None:
            labels[labels == processor.tokenizer.pad_token_id] = -100

        item["labels"] = labels

        return item

In [8]:
def vlm_collate_fn(batch):
    collated = {}

    for key in batch[0].keys():
        values = [item[key] for item in batch]

        if key in ["input_ids", "attention_mask", "labels"]:
            if key == "input_ids":
                pad_value = processor.tokenizer.pad_token_id
            elif key == "attention_mask":
                pad_value = 0
            else:
                pad_value = -100

            collated[key] = torch.nn.utils.rnn.pad_sequence(
                values,
                batch_first=True,
                padding_value=pad_value
            )
        else:
            try:
                collated[key] = torch.stack(values)
            except:
                collated[key] = values

    return collated

In [9]:
train_dataset = ScienceVLMDataset(train_df)
val_dataset = ScienceVLMDataset(val_df)

sample = train_dataset[0]

for k, v in sample.items():
    if hasattr(v, "shape"):
        print(k, v.shape)
    else:
        print(k, type(v))

print("train_dataset length:", len(train_dataset))
print("val_dataset length:", len(val_dataset))

pixel_values torch.Size([17, 3, 512, 512])
pixel_attention_mask torch.Size([17, 512, 512])
input_ids torch.Size([1455])
attention_mask torch.Size([1455])
labels torch.Size([1455])
train_dataset length: 3109
val_dataset length: 1048


In [ ]:
import gc
import torch
from transformers import TrainingArguments, Trainer

gc.collect()
torch.cuda.empty_cache()

model.config.use_cache = False
model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir="/content/smolvlm_lora_mlp_metadata",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=3e-5,
    num_train_epochs=2,

    logging_steps=10,
    save_steps=100,
    save_total_limit=1,

    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",

    max_grad_norm=0.3,

    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=vlm_collate_fn,
)

trainer.train()

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,10.670597
20,5.293576
30,1.638211
40,1.091877
50,1.063179
60,0.981144
70,0.911194
80,0.897344
90,0.796339
100,0.880244


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/kagglefinal/smolvlm_lora_mlp_metadata_ep3"

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print("Saved to:", SAVE_DIR)

In [ ]:
from tqdm.auto import tqdm
from PIL import Image
import pandas as pd
import torch
import torch.nn.functional as F
import numpy as np
from google.colab import files

@torch.no_grad()
def score_candidate_answer(row, candidate_idx):
    img_path = resolve_image_path(row["image_path"])
    image = Image.open(img_path).convert("RGB")

    prompt_text = build_prompt_text(row)
    full_text = prompt_text + str(candidate_idx)

    prompt_inputs = processor(
        text=prompt_text,
        images=[image],
        return_tensors="pt",
        padding=False,
    ).to(model.device)

    full_inputs = processor(
        text=full_text,
        images=[image],
        return_tensors="pt",
        padding=False,
    ).to(model.device)

    prompt_len = prompt_inputs["input_ids"].shape[1]

    outputs = model(**full_inputs)
    logits = outputs.logits

    input_ids = full_inputs["input_ids"]

    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]

    candidate_start = max(prompt_len - 1, 0)

    cand_logits = shift_logits[:, candidate_start:, :]
    cand_labels = shift_labels[:, candidate_start:]

    log_probs = F.log_softmax(cand_logits, dim=-1)
    token_log_probs = log_probs.gather(
        dim=-1,
        index=cand_labels.unsqueeze(-1)
    ).squeeze(-1)

    score = token_log_probs.mean().item()
    return score


@torch.no_grad()
def predict_one_mc_likelihood(row):
    scores = []

    for candidate_idx in range(int(row["num_choices"])):
        score = score_candidate_answer(row, candidate_idx)
        scores.append(score)

    pred = int(np.argmax(scores))
    return pred, scores


model.eval()

test_preds = []
test_scores = []

for i in tqdm(range(len(test_df))):
    pred, scores = predict_one_mc_likelihood(test_df.iloc[i])
    test_preds.append(pred)
    test_scores.append(scores)

submission = sample_submission.copy()
submission["answer"] = test_preds

OUT_PATH = "/content/submission_lora_mlp_metadata_mcloglik.csv"
submission.to_csv(OUT_PATH, index=False)

print(submission.head())
print(submission["answer"].value_counts().sort_index())

sub = pd.read_csv(OUT_PATH)

assert list(sub.columns) == ["id", "answer"]
assert len(sub) == len(test_df)
assert sub["id"].tolist() == sample_submission["id"].tolist()
assert sub["answer"].notna().all()

merged = sub.merge(test_df[["id", "num_choices"]], on="id")
assert (merged["answer"] >= 0).all()
assert (merged["answer"] < merged["num_choices"]).all()

print("Valid submission saved to:", OUT_PATH)

files.download(OUT_PATH)